# Candidate Recommendation System Project

# Part 1 - Data Collection & Preprocessing


**Dataset used:** [Resume Dataset (Kaggle)](https://www.kaggle.com/datasets/saugataroyarghya/resume-dataset) - a real-world dataset containing 9,544 candidate–job pairings across 28 unique job roles, with structured fields for education, skills, work experience, and job requirements, plus a pre-computed `matched_score` for each pairing.

**Notebook content:**
1. Loads the raw dataset and inspects its structure
2. Selects a diverse sample of candidates across multiple job categories
3. Splits the data into two separate datasets - **candidates** and **job descriptions**
4. Merges relevant fields into a single text document per candidate / per job
5. Cleans and preprocesses the text (lowercasing, removing punctuation/numbers, tokenization, stopword removal, lemmatization)
6. Saves the cleaned text files, ready for vectorization in Part 2
7. Saves the `matched_score` values separately as a ground-truth reference for Part 4 (Evaluation)


## 1. Imports and Setup

In [1]:
import os
import re
import ast
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [2]:
# Download required NLTK resources (only needs to run once. run once and comment)

# nltk.download('stopwords')
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

In [3]:
pd.set_option('display.max_colwidth', 150)

## 2. Load the Raw Dataset

The raw CSV has **9,544 rows and 35 columns**. Each row represents one candidate paired with one job they applied for. The columns fall into three groups:

- **Candidate-side columns:** `career_objective`, `skills`, `educational_institution_name`, `degree_names`,
  `major_field_of_studies`, `professional_company_names`, `positions`, `languages`, `certification_skills`, etc.
- **Job-side columns:** `job_position_name`, `educationaL_requirements`, `experiencere_requirement`,
  `skills_required`, `responsibilities.1`
- **Evaluation column:** `matched_score` - a pre-computed similarity score between the candidate and the job (0 to ~0.97)

Note: the original file has a UTF-8 BOM (byte-order-mark) artifact on the `job_position_name` column header, which we clean up during loading.


In [4]:
RAW_PATH = "../data/raw/resume_data.csv"

df = pd.read_csv(RAW_PATH)

# Fix the BOM artifact on the job_position_name column header
df = df.rename(columns={"\ufeffjob_position_name": "job_position_name"})

print("Shape:", df.shape)
df.head(3)


Shape: (9544, 35)


,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database warehouse manager with robust experience in handling all kinds of data. I have also used multiple cloud in...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapreduce', 'Spark', 'Java', 'Machine Learning', 'Cloud', 'Hdfs', 'YARN', 'Core Java', 'Data Science', '...","['The Amity School of Engineering & Technology (ASET), Noida']",['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a reputed university.,At least 1 year,NaN,Technical Support\r\nTroubleshooting\r\nCollaboration\r\nDocumentation\r\nSystem Monitoring\r\nSoftware Deployment\r\nTraining & Mentorship\r\nInd...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and junior data scientist. Experienced in creating meaningful data dashboards and evaluation models.,"['Data Analysis', 'Data Analytics', 'Business Analysis', 'R', 'SAS', 'PowerBi', 'Tableau', 'Data Visualization', 'Business Analytics', 'Machine Le...","['Delhi University - Hansraj College', 'Delhi University - Hansraj College']","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in any relevant discipline from a reputed University,At least 5 year(s),NaN,Machine Learning Leadership\r\nCross-Functional Collaboration\r\nStrategy Development\r\nML/NLP Infrastructure\r\nPrototype Transformation\r\nML S...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', 'Deep Learning', 'Risk Assessment', 'Requirement Gathering', 'Application Support', 'JavaScript', 'Py...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],...,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, Hygiene Products",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\r\nBrand Visibility, Sales Targets\r\nField Marketing, Campaigns, Product Distribution\r\nBrand Head\r\nExcel, KPIs Trac...",Brand Promotion\r\nCampaign Management\r\nField Supervision\r\nMerchandising\r\npromotional activities\r\nTrade Marketing,0.416667


## 3. Select a Diverse Sample

The dataset only has **28 unique job titles**, each repeated ~340 times (once per candidate who applied). To keep the project scope realistic and manageable (per the brief: 10–15 CVs, 3-5 job descriptions), we select **4 job roles from different fields** so the recommendation results are meaningful and easy to demonstrate, rather than picking near-duplicate roles:

- **Senior Software Engineer** (Technology)
- **HR Officer** (Human Resources)
- **Civil Engineer** (Engineering / Construction)
- **Business Development Executive** (Business / Sales)

For each role, we take a sample of **3 candidates**, giving us **12 candidates total** across **4 jobs** -
within the brief's suggested range.


In [5]:
TARGET_JOBS = [
    "Senior Software Engineer",
    "HR Officer",
    "Civil Engineer",
    "Business Development Executive",
]

CANDIDATES_PER_JOB = 3

sample_frames = []
for job_title in TARGET_JOBS:
    subset = df[df["job_position_name"] == job_title].sample(
        n=CANDIDATES_PER_JOB, random_state=42
    )
    sample_frames.append(subset)

sample_df = pd.concat(sample_frames).reset_index(drop=True)

print(f"Selected {len(sample_df)} candidate rows across {sample_df['job_position_name'].nunique()} unique jobs")
sample_df[["job_position_name", "matched_score"]]


Selected 12 candidate rows across 4 unique jobs


,job_position_name,matched_score
0,Senior Software Engineer,0.683333
1,Senior Software Engineer,0.850000
2,Senior Software Engineer,0.816667
3,HR Officer,0.683333
4,HR Officer,0.650000
5,HR Officer,0.716667
6,Civil Engineer,0.540000
7,Civil Engineer,0.350000
8,Civil Engineer,0.250000
9,Business Development Executive,0.683333


## 4. Handle List-Formatted Columns

Several columns (`skills`, `degree_names`, `major_field_of_studies`, `positions`, `languages`, `certification_skills`) are stored as **Python list literals written as strings**, e.g.:

```
"['Big Data', 'Hadoop', 'Hive', 'Python', 'Spark']"
```

These need to be parsed back into actual lists and joined into plain text before they can be merged with the other text fields. We use `ast.literal_eval` for safe parsing, with a fallback for missing or malformed values.


In [6]:
def parse_list_field(value):
    """Safely parse a string that looks like a Python list into space-separated text."""
    if pd.isna(value):
        return ""
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            # Filter out None entries and join into a single string
            return " ".join(str(item) for item in parsed if item is not None)
        return str(parsed)
    except (ValueError, SyntaxError):
        return str(value)


def safe_text(value):
    """Return an empty string for missing values, otherwise the value as text."""
    if pd.isna(value):
        return ""
    return str(value)


# Quick test
print(parse_list_field("['Big Data', 'Hadoop', 'Hive', 'Python']"))
print(parse_list_field(None))


Big Data Hadoop Hive Python



## 5. Split the Data into Candidates and Job Descriptions

We build two separate documents per row:

**Candidate document** - combines: `career_objective`, `skills`, `degree_names`, `major_field_of_studies`, `positions`, `languages`, `certification_skills`

**Job document** - combines: `job_position_name`, `educationaL_requirements`, `experiencere_requirement`, `skills_required`, `responsibilities.1`

Note: the `responsibilities` column (candidate side) was found to be **identical** to `responsibilities.1` (job side) for every row in this dataset - it appears to describe the job's responsibilities rather than the candidate's own work history, so it is excluded from the candidate document to avoid leaking job information directly into the CV text.

Job descriptions are then **de-duplicated** by job title, since each of our 4 roles currently appears 3 times (once per sampled candidate).


In [7]:
def build_candidate_text(row):
    parts = [
        safe_text(row["career_objective"]),
        parse_list_field(row["skills"]),
        parse_list_field(row["degree_names"]),
        parse_list_field(row["major_field_of_studies"]),
        parse_list_field(row["positions"]),
        parse_list_field(row["languages"]),
        parse_list_field(row["certification_skills"]),
    ]
    return " ".join(p for p in parts if p)


def build_job_text(row):
    parts = [
        safe_text(row["job_position_name"]),
        safe_text(row["educationaL_requirements"]),
        safe_text(row["experiencere_requirement"]),
        safe_text(row["skills_required"]),
        safe_text(row["responsibilities.1"]),
    ]
    return " ".join(p for p in parts if p)


sample_df["candidate_text"] = sample_df.apply(build_candidate_text, axis=1)
sample_df["job_text"] = sample_df.apply(build_job_text, axis=1)

# Give each candidate a simple ID
sample_df["candidate_id"] = [f"candidate_{i+1:02d}" for i in range(len(sample_df))]

# De-duplicate job descriptions by job title
jobs_df = sample_df.drop_duplicates(subset="job_position_name")[
    ["job_position_name", "job_text"]
].reset_index(drop=True)
jobs_df["job_id"] = [f"job_{i+1:02d}" for i in range(len(jobs_df))]

print(f"Candidates: {len(sample_df)}")
print(f"Unique jobs: {len(jobs_df)}")


Candidates: 12
Unique jobs: 4


In [8]:
print("=== Sample candidate document (raw, before cleaning) ===")
print(sample_df.loc[0, "candidate_id"], "-", sample_df.loc[0, "job_position_name"])
print(sample_df.loc[0, "candidate_text"][:500])


=== Sample candidate document (raw, before cleaning) ===
candidate_01 - Senior Software Engineer
Seeking challenging opportunity in the field of data science and data analytics where I can utilize my skills and knowledge to contribute for the growth of the organization Python MySQL Tensorflow Keras Machine Learning Deep Learning B.Tech Metallurgy Intern


In [9]:
print("=== Sample job document (raw, before cleaning) ===")
print(jobs_df.loc[0, "job_id"], "-", jobs_df.loc[0, "job_position_name"])
print(jobs_df.loc[0, "job_text"][:500])


=== Sample job document (raw, before cleaning) ===
job_01 - Senior Software Engineer
Senior Software Engineer B.Sc in Computer Science & Engineering from a reputed university. At least 1 year Technical Support
Troubleshooting
Collaboration
Documentation
System Monitoring
Software Deployment
Training & Mentorship
Industry Trends
Field Visits







## 6. Text Cleaning Function

Both candidate documents and job documents are cleaned using the **exact same pipeline**, so that the resulting text is directly comparable when vectorized in Part 2. Steps:

1. Lowercase all text
2. Remove punctuation and numbers
3. Tokenize into words
4. Remove English stopwords (e.g. "the", "is", "and")
5. Remove leftover single-character tokens
6. Lemmatize each remaining word (e.g. "developing" → "developing" → "develop")


In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()

    # Remove punctuation and numbers, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and short tokens, then lemmatize
    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    return ' '.join(cleaned_tokens)


# Quick test
sample_test = "The candidate has 5+ years of experience in Software Engineering, developing REST APIs!"
print("Before:", sample_test)
print("After: ", clean_text(sample_test))


Before: The candidate has 5+ years of experience in Software Engineering, developing REST APIs!
After:  candidate year experience software engineering developing rest apis


After:  candidate year experience software engineering developing rest apis


## 7. Apply Cleaning to Candidates and Jobs

In [11]:
sample_df["candidate_text_clean"] = sample_df["candidate_text"].apply(clean_text)
jobs_df["job_text_clean"] = jobs_df["job_text"].apply(clean_text)

print("Cleaning complete.")
print(f"Cleaned {len(sample_df)} candidate documents")
print(f"Cleaned {len(jobs_df)} job documents")


Cleaning complete.
Cleaned 12 candidate documents
Cleaned 4 job documents


In [12]:
print("=== Sample cleaned candidate document ===")
print(sample_df.loc[0, "candidate_id"])
print(sample_df.loc[0, "candidate_text_clean"][:500])
print()
print("=== Sample cleaned job document ===")
print(jobs_df.loc[0, "job_id"])
print(jobs_df.loc[0, "job_text_clean"][:500])


=== Sample cleaned candidate document ===
candidate_01
seeking challenging opportunity field data science data analytics utilize skill knowledge contribute growth organization python mysql tensorflow kera machine learning deep learning tech metallurgy intern

=== Sample cleaned job document ===
job_01
senior software engineer sc computer science engineering reputed university least year technical support troubleshooting collaboration documentation system monitoring software deployment training mentorship industry trend field visit


## 8. Save Cleaned Text Files

Each candidate and job becomes its own `.txt` file, following the same structure the whole team is using (`/data/cleaned_cvs`, `/data/cleaned_jobs`), ready for Part 2 (Vectorization) to load directly.


In [13]:
CLEANED_CV_DIR = "../data/cleaned_cvs"
CLEANED_JOB_DIR = "../data/cleaned_jobs"

os.makedirs(CLEANED_CV_DIR, exist_ok=True)
os.makedirs(CLEANED_JOB_DIR, exist_ok=True)

for _, row in sample_df.iterrows():
    filepath = os.path.join(CLEANED_CV_DIR, f"{row['candidate_id']}.txt")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(row["candidate_text_clean"])

for _, row in jobs_df.iterrows():
    filepath = os.path.join(CLEANED_JOB_DIR, f"{row['job_id']}.txt")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(row["job_text_clean"])

print(f"Saved {len(sample_df)} cleaned candidate files to '{CLEANED_CV_DIR}'")
print(f"Saved {len(jobs_df)} cleaned job files to '{CLEANED_JOB_DIR}'")


Saved 12 cleaned candidate files to '../data/cleaned_cvs'
Saved 4 cleaned job files to '../data/cleaned_jobs'


## 9. Save Ground Truth Reference (for Part 4 - Evaluation)

The dataset's `matched_score` field gives a pre-computed similarity score for each candidate - job pairing in the original data. This is **not** used in preprocessing or vectorization - it is set aside for whoever handles Part 4 (Evaluation), so the team's own similarity scores can be compared against a real reference instead of having to invent ground truth manually.


In [14]:
ground_truth = sample_df[["candidate_id", "job_position_name", "matched_score"]].copy()
ground_truth = ground_truth.merge(
    jobs_df[["job_position_name", "job_id"]], on="job_position_name", how="left"
)
ground_truth = ground_truth[["candidate_id", "job_id", "job_position_name", "matched_score"]]

GROUND_TRUTH_PATH = "../data/ground_truth.csv"
ground_truth.to_csv(GROUND_TRUTH_PATH, index=False)

print(f"Saved ground truth reference to '{GROUND_TRUTH_PATH}'")
ground_truth


Saved ground truth reference to '../data/ground_truth.csv'


,candidate_id,job_id,job_position_name,matched_score
0,candidate_01,job_01,Senior Software Engineer,0.683333
1,candidate_02,job_01,Senior Software Engineer,0.850000
2,candidate_03,job_01,Senior Software Engineer,0.816667
3,candidate_04,job_02,HR Officer,0.683333
4,candidate_05,job_02,HR Officer,0.650000
5,candidate_06,job_02,HR Officer,0.716667
6,candidate_07,job_03,Civil Engineer,0.540000
7,candidate_08,job_03,Civil Engineer,0.350000
8,candidate_09,job_03,Civil Engineer,0.250000
9,candidate_10,job_04,Business Development Executive,0.683333


## 10. Preprocessing Pipeline Summary (for the Report)

**Dataset:** Resume Dataset (Kaggle) - 9,544 real candidate - job pairings, 35 columns, 28 unique job roles.

**Sampling:** Selected a diverse sample of 4 job roles spanning different fields (Technology, HR, Engineering, Business) with 3 candidates per role - 12 candidates and 4 unique jobs total, in line with the project brief's suggested range (10 - 15 CVs, 3 - 5 jobs).

**Splitting:** The raw dataset pairs one candidate with one job per row. Candidate-side and job-side columns were separated into two datasets, and job postings were de-duplicated by title since the same job appears multiple times (once per applicant).

**Merging:** Several fields (`skills`, `degree_names`, `major_field_of_studies`, `positions`, `languages`, `certification_skills`) were stored as Python list literals inside strings and were parsed with `ast.literal_eval` before being joined into plain text. All relevant fields were then combined into a single text document per candidate and per job.

**Cleaning steps applied identically to both candidates and jobs:**
1. Lowercased all text
2. Removed punctuation and numbers using regex
3. Tokenized text into individual words using NLTK's `word_tokenize`
4. Removed English stopwords using NLTK's stopword list
5. Removed leftover single-character tokens
6. Lemmatized each remaining word using NLTK's `WordNetLemmatizer`

**Output:**
- 12 cleaned candidate text files in `data/cleaned_cvs/`
- 4 cleaned job description text files in `data/cleaned_jobs/`
- 1 ground truth reference file (`data/ground_truth.csv`) containing the original `matched_score` for each candidate - job pairing, reserved for Part 4 (Evaluation)

**Limitations noted:**
- The dataset's `responsibilities` (candidate) and `responsibilities.1` (job) columns were found to be identical for every row, suggesting the column describes the job's responsibilities rather than the candidate's actual work history - it was therefore excluded from the candidate document.
- Some fields had missing values (e.g. `career_objective` missing in ~50% of rows); these were treated as empty strings during merging rather than dropped, to avoid losing candidates from the sample.
